# SearchSense
## 04 — Query Expansion

**Goal:** Build a model that predicts the full intended query from a partial input,
enabling real-time intent prediction as the user types.  
**Author:** Ramya Manasa Amancherla


In [2]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss
import pickle
import os

# Sample queries representing realistic search intent across surfaces
# In production this would come from query logs
queries = [
    "best laptop under 1000", "cheap flights to miami", "running shoes for women",
    "wireless noise cancelling headphones", "best coffee maker 2024",
    "yoga mat thick non slip", "skincare routine for dry skin", "whey protein powder",
    "gaming chair ergonomic", "winter jacket men", "air fryer recipes",
    "standing desk adjustable", "bluetooth speaker waterproof", "hiking boots waterproof",
    "polarized sunglasses men", "smartwatch fitness tracker", "laptop bag leather",
    "water bottle insulated", "iphone 15 case", "desk lamp led",
    "best running shoes", "cheap hotels new york", "mens winter boots",
    "wireless earbuds", "instant pot recipes", "office chair ergonomic",
    "best gaming laptop", "protein bars low sugar", "yoga pants women",
    "air purifier for bedroom"
]

print(f"Total queries: {len(queries)}")
print(f"Sample queries: {queries[:5]}")

/Users/ramyaamancherla/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Total queries: 30
Sample queries: ['best laptop under 1000', 'cheap flights to miami', 'running shoes for women', 'wireless noise cancelling headphones', 'best coffee maker 2024']


## 1. Generate Partial Query Training Pairs
For each full query, generate all possible partial versions to simulate real typing behavior.

In [3]:
# Generate partial query pairs from full queries
# Each full query produces multiple partial versions of increasing length
pairs = []

for full_query in queries:
    words = full_query.split()
    # Generate partials at word boundaries
    for i in range(1, len(words) + 1):
        partial = ' '.join(words[:i])
        pairs.append({
            'partial_query': partial,
            'full_query': full_query,
            'completeness': len(partial) / len(full_query)
        })
    
    # Also generate character-level partials for the first word
    first_word = words[0]
    for i in range(1, len(first_word)):
        partial = first_word[:i]
        pairs.append({
            'partial_query': partial,
            'full_query': full_query,
            'completeness': len(partial) / len(full_query)
        })

pairs_df = pd.DataFrame(pairs).drop_duplicates(subset=['partial_query', 'full_query'])
pairs_df = pairs_df.sort_values('completeness')

print(f"Total partial-full pairs: {len(pairs_df)}")
print(f"\nSample pairs:")
print(pairs_df[['partial_query', 'full_query', 'completeness']].head(15).to_string(index=False))

Total partial-full pairs: 245

Sample pairs:
partial_query                           full_query  completeness
            w wireless noise cancelling headphones      0.027778
            s        skincare routine for dry skin      0.034483
            b         bluetooth speaker waterproof      0.035714
            s           smartwatch fitness tracker      0.038462
            s             standing desk adjustable      0.041667
            a             air purifier for bedroom      0.041667
            p             polarized sunglasses men      0.041667
            h              hiking boots waterproof      0.043478
            r              running shoes for women      0.043478
            y              yoga mat thick non slip      0.043478
            p               protein bars low sugar      0.045455
            g               gaming chair ergonomic      0.045455
            b               best laptop under 1000      0.045455
            o               office chair ergo

## 2. Build Semantic Search Index
Embed all full queries using a sentence transformer and index them with FAISS for fast similarity search.

In [4]:
# Load pretrained sentence transformer
# No fine-tuning needed, the model already understands semantic similarity
print("Loading sentence transformer model...")
model = SentenceTransformer('all-MiniLM-L6-v2')

# Embed all full queries
print("Embedding full queries...")
full_query_embeddings = model.encode(queries, show_progress_bar=True)

print(f"\nEmbedding shape: {full_query_embeddings.shape}")
print(f"Each query is represented as a {full_query_embeddings.shape[1]}-dimensional vector")

Loading sentence transformer model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding full queries...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Embedding shape: (30, 384)
Each query is represented as a 384-dimensional vector


## 3. Build FAISS Index
Index the embeddings for fast nearest-neighbor search at inference time.

In [5]:
# Build FAISS index for fast similarity search
dimension = full_query_embeddings.shape[1]

# L2 distance index
index = faiss.IndexFlatL2(dimension)
index.add(full_query_embeddings.astype(np.float32))

print(f"FAISS index built")
print(f"Total vectors indexed: {index.ntotal}")
print(f"Vector dimension: {dimension}")

# Test the index with a partial query
test_partial = "best lapt"
test_embedding = model.encode([test_partial])
distances, indices = index.search(test_embedding.astype(np.float32), k=3)

print(f"\nTest: '{test_partial}'")
print(f"Top 3 predicted full queries:")
for i, (dist, idx) in enumerate(zip(distances[0], indices[0])):
    print(f"  {i+1}. '{queries[idx]}' (distance: {dist:.4f})")

FAISS index built
Total vectors indexed: 30
Vector dimension: 384

Test: 'best lapt'
Top 3 predicted full queries:
  1. 'best laptop under 1000' (distance: 1.2701)
  2. 'best gaming laptop' (distance: 1.3340)
  3. 'best coffee maker 2024' (distance: 1.4823)


### Key Findings
The sentence transformer correctly resolves "best lapt" to "best laptop under 1000" as the top 
prediction, with "best gaming laptop" as a close second. This demonstrates that semantic embeddings 
capture typing intent even from incomplete input. The model requires no fine-tuning because the 
pretrained representations already encode meaningful similarity between partial and complete queries.

## 4. Save Index and Model


In [6]:
# Save FAISS index, query list, and sentence transformer for inference
faiss.write_index(index, "backend/faiss_index.bin")

with open("backend/queries.pkl", "wb") as f:
    pickle.dump(queries, f)

# Save model name for loading at inference time
with open("backend/embedding_model_name.pkl", "wb") as f:
    pickle.dump('all-MiniLM-L6-v2', f)

print("Saved:")
print("  backend/faiss_index.bin")
print("  backend/queries.pkl")
print("  backend/embedding_model_name.pkl")

Saved:
  backend/faiss_index.bin
  backend/queries.pkl
  backend/embedding_model_name.pkl
